In [ ]:
# Overlay check before any removal: discovery on the first 60 s of each video; only text results leave Kaggle
import glob, os, subprocess, zipfile
VERSION = '0.6.2'
DATASET = 'huohua-studio-test-data'
W = '/kaggle/working'
import shutil
z = glob.glob(f'/kaggle/input/**/wmr-{VERSION}.zip', recursive=True)
d = glob.glob(f'/kaggle/input/**/wmr-{VERSION}/wmr/track.py', recursive=True)
if z:
    zipfile.ZipFile(z[0]).extractall(W)
else:   # Kaggle unpacks zips when it mounts a dataset
    shutil.copytree(os.path.dirname(os.path.dirname(d[0])), f'{W}/wmr-{VERSION}', dirs_exist_ok=True)
IN_DIR = next(b for b, d, f in os.walk('/kaggle/input', followlinks=True)
              if DATASET in b and any(x.lower().endswith('.mp4') for x in f))
videos = sorted(f for f in os.listdir(IN_DIR) if f.lower().endswith('.mp4'))
print(IN_DIR, videos)

In [ ]:
!bash /kaggle/working/wmr-{VERSION}/scripts/setup_kaggle.sh 2>&1 | tail -2

In [ ]:
# Removal on videos whose overlays were checked (072q-1: only 'Huohua Studio' in the first 60 s).
# v12 / v27 also carry third-party creator and site marks, so they are not processed.
# The output video stays on Kaggle for the owner to review; only results.txt and prof.txt are fetched.
import re, threading, time, collections
R = open(f'{W}/results.txt', 'w')
def say(s):
    print(s, flush=True); R.write(s + '\n'); R.flush()
gpu, stop = [], threading.Event()
def poll():   # GPU utilisation every 2 s, both T4s
    while not stop.is_set():
        o = subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout
        gpu.append((time.time(), [tuple(int(x) for x in l.split(',')) for l in o.strip().splitlines()]))
        stop.wait(2)
threading.Thread(target=poll, daemon=True).start()
for name, text in {'072q-1.watermarked.mp4': 'Huohua Studio'}.items():
    out = f"{W}/{name.split('.')[0]}.wmr.mp4"
    open(f'{W}/texts.txt', 'w').write(text + '\n')
    t = time.time()
    p = subprocess.Popen(['python', '-m', 'wmr', 'run', os.path.join(IN_DIR, name), '-o', out, '--texts', f'{W}/texts.txt'],
                         cwd=f'{W}/wmr-{VERSION}', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'WMR_PROF': f'{W}/prof.txt'})
    for l in p.stdout:
        if any(k in l for k in ('[analyze]', '[remove]', '[verify]', '[run]', 'track ', 'note:', 'warning', 'Error', 'Traceback')):
            say(f'  t={time.time() - t:6.0f}  ' + l.strip()[:420])
    say(f'=== {name}: {time.time() - t:.0f} s wall, exit {p.wait()}')
    if os.path.exists(out):
        say(f'  output {os.path.getsize(out) / 1e6:.0f} MB vs source {os.path.getsize(os.path.join(IN_DIR, name)) / 1e6:.0f} MB')
stop.set()
# GPU utilisation per phase (phase epochs from prof.txt)
for m in re.finditer(r'##### (.+?): .*?epoch (\d+)\.\.(\d+)', open(f'{W}/prof.txt').read()):
    a, b = int(m.group(2)), int(m.group(3))
    rows = [g for ts, gs in gpu if a <= ts <= b for g in gs]
    by = collections.defaultdict(list)
    for i, u, mem in rows:
        by[i].append((u, mem))
    say(f'  gpu {m.group(1)}: ' + ', '.join(f'GPU{i} util {sum(u for u, _ in v) / len(v):.0f}% (max mem {max(m_ for _, m_ in v)} MiB)'
                                      for i, v in sorted(by.items())))
R.close()


In [ ]:
# keep results.txt, prof.txt and the output video (for review on Kaggle)
import shutil
for x in os.listdir(W):
    if x not in ('results.txt', 'prof.txt') and not x.endswith('.wmr.mp4'):
        p = os.path.join(W, x); shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
!ls -la /kaggle/working
